In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from statsmodels.tsa.stattools import pacf
import numpy as np
import pandas as pd

In [15]:
import os
import sys
current_dir = os.path.dirname(os.path.realpath(__name__))
parent_dir = os.path.dirname(current_dir)
print(parent_dir)
sys.path.append(parent_dir)
from src.data.data_loader import DataLoader
from src.data.time_series import get_ar_terms, get_lag_dict

C:\Users\jonah.eisen\dsi\global-stock-index-ml-classification


In [16]:
X, y = dl.prepare_X_y()
X.head()

,Volume,LowProportion,HighProportion,Delta Close_(t-1),Delta Close_(t-2),Delta Close_(t-3),Delta Close_(t-4),Delta Close_(t-6),Delta Close_(t-7),Delta Close_(t-8),Delta Close_(t-9),Delta Close_(t-11),Delta Close_(t-14),Delta Close_(t-15),Delta Close_(t-16),Delta Close_(t-21),Delta Close_(t-22),Delta Close_(t-23),Delta Close_(t-29)
Date,,,,,,,,,,,,,,,,,,,
2003-02-24,1.219200e+09,0.983069,1.000054,-24.629883,23.699707,-74.890136,33.700195,80.040039,-55.439941,-83.060059,-4.970215,-94.199707,-44.220215,100.359863,2.720215,-82.790039,-20.619629,-145.529786,-0.830078
2003-02-25,1.495500e+09,0.982515,1.002097,-73.790039,-24.629883,23.699707,-74.890136,152.479980,80.040039,-55.439941,-83.060059,-24.540039,-76.930176,-44.220215,100.359863,-208.700195,-82.790039,-20.619629,23.860351
2003-02-26,1.382300e+09,0.986621,1.000000,-52.240235,-73.790039,-24.629883,23.699707,33.700195,152.479980,80.040039,-55.439941,-4.970215,-67.170410,-76.930176,-44.220215,-39.199707,-208.700195,-82.790039,-37.759766
2003-02-27,1.297100e+09,0.998593,1.012544,-19.840332,-52.240235,-73.790039,-24.629883,-74.890136,33.700195,152.479980,80.040039,-83.060059,-94.199707,-67.170410,-76.930176,78.979981,-39.199707,-208.700195,-68.390136
2003-02-28,1.311700e+09,1.000000,1.010189,62.069824,-19.840332,-52.240235,-73.790039,23.699707,-74.890136,33.700195,152.479980,-55.439941,-24.540039,-94.199707,-67.170410,-56.540039,78.979981,-39.199707,-62.940429


In [17]:
y.head()

Date
2003-02-24    0
2003-02-25    0
2003-02-26    0
2003-02-27    1
2003-02-28    1
dtype: int64

In [18]:
val_size = 0.2
X_train, X_test = DataLoader.time_split_2D(X)
X_train, X_val = DataLoader.time_split_2D(X_train)
y_train, y_test = DataLoader.time_split_1D(y)
y_train, y_val = DataLoader.time_split_1D(y_train)

In [19]:
#X_train  = X_train.rolling(window=7, center=False).mean().dropna()
#y_train  = y_train.rolling(window=7, center=False).mean().dropna()
y_train = DataLoader.quantize_delta_close(y_train)
y_val = DataLoader.quantize_delta_close(y_val)

In [20]:
X_train.head()
y_train.head()

Date
2003-02-24    0
2003-02-25    0
2003-02-26    0
2003-02-27    1
2003-02-28    1
dtype: int64

In [21]:
rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
y_prob = rf.predict_proba(X_val)[:,1]



In [22]:
print("Random Forest Accuracy:", rf.score(X_val, y_val))
print(classification_report(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_prob))

Random Forest Accuracy: 0.686141304347826
              precision    recall  f1-score   support

           0       0.70      0.56      0.62       340
           1       0.68      0.80      0.73       396

    accuracy                           0.69       736
   macro avg       0.69      0.68      0.68       736
weighted avg       0.69      0.69      0.68       736

ROC-AUC: 0.7546865715983363


In [23]:
y_pred_train = rf.predict(X_train)
y_prob_train = rf.predict_proba(X_train)[:,1]
print("Random Forest Accuracy:", rf.score(X_train, y_train))
print(classification_report(y_train, y_pred_train))
print("ROC-AUC:", roc_auc_score(y_train, y_prob_train))

Random Forest Accuracy: 0.751869476546567
              precision    recall  f1-score   support

           0       0.76      0.66      0.70      1321
           1       0.75      0.83      0.79      1621

    accuracy                           0.75      2942
   macro avg       0.75      0.74      0.75      2942
weighted avg       0.75      0.75      0.75      2942

ROC-AUC: 0.838303661117029
